# Multi-Image RAG — Google Colab

Run the full **Multimodal Image RAG** pipeline on Colab GPU.

| Component | Covers original file | Role |
|-----------|---------------------|------|
| **CLIP** (ViT-B/32) | `clip_service.py` + `clip_encoder.py` | Image & text → 512-dim embeddings |
| **ChromaDB** | `chroma_service.py` | Cosine-HNSW vector store |
| **Image utilities** | `image_utils.py` | Validation, base64, safe load |
| **BLIP** *(default)* | — | Local image captioning / analysis |
| **Ollama + LLaVA** *(optional)* | `ollama_service.py` | Instruction-following vision LLM |
| **Gradio** | `api/routes.py` (all endpoints) | Upload, search, analyse, manage |

### Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells top-to-bottom *(Cells 1 → 11)*
3. Cell 11 prints a public Gradio link — open it in any browser

> **Optional LLaVA**: run Cell 7 (Ollama install), then set `USE_OLLAMA = True` in Cell 6 and re-run cells 6 & 8.

In [ ]:
# Cell 1 — Install dependencies & verify GPU
!pip install -q chromadb gradio transformers torch torchvision pillow accelerate sentencepiece

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU detected. Enable: Runtime > Change runtime type > T4 GPU')
print('Installation complete!')

In [ ]:
# Cell 2 — Config, directories, logging
# Mirrors: utils/config.py  (Settings class)
#        + utils/logger.py  (setup_logging)

import os
import logging

# ── Paths — Option A: local Colab (lost when session ends) ───────────────────
UPLOAD_DIR  = '/content/uploads'
CHROMA_PATH = '/content/vectordb_store'

# ── Option B: Google Drive (uncomment to persist across sessions) ─────────────
# from google.colab import drive
# drive.mount('/content/drive')
# UPLOAD_DIR  = '/content/drive/MyDrive/MultiImageRAG/uploads'
# CHROMA_PATH = '/content/drive/MyDrive/MultiImageRAG/vectordb_store'

# ── Settings (mirrors Settings class in config.py) ────────────────────────────
CLIP_MODEL_NAME    = 'openai/clip-vit-base-patch32'
CHROMA_COLLECTION  = 'image_rag'
TOP_K_DEFAULT      = 5
MAX_UPLOAD_MB      = 10
ALLOWED_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.webp'}
ALLOWED_FORMATS    = {'PNG', 'JPEG', 'JPG', 'WEBP'}
OLLAMA_URL         = 'http://localhost:11434'
OLLAMA_MODEL       = 'llava'
OLLAMA_TIMEOUT     = 120   # seconds — mirrors ollama_timeout_seconds
OLLAMA_RETRIES     = 3     # mirrors ollama_max_retries

os.makedirs(UPLOAD_DIR,  exist_ok=True)
os.makedirs(CHROMA_PATH, exist_ok=True)

# ── Logging (mirrors setup_logging() in logger.py) ────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
)
log = logging.getLogger('image_rag')

print(f'Upload dir  : {UPLOAD_DIR}')
print(f'ChromaDB dir: {CHROMA_PATH}')
print('Config ready.')

In [ ]:
# Cell 3 — Image utilities
# Full mirror of: backend/utils/image_utils.py
# Functions: validate_image, load_image, image_to_base64,
#            now_iso, save_upload, ensure_upload_dir

import base64
import os
import uuid
from datetime import datetime
from PIL import Image as PILImage


def validate_image_file(file_path: str) -> None:
    '''Mirrors validate_image() — checks allowed extension and max file size.'''
    ext = os.path.splitext(file_path)[1].lower()
    if ext not in ALLOWED_EXTENSIONS:
        raise ValueError(f'Unsupported format: {ext}. Allowed: PNG, JPEG, WEBP.')
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    if size_mb > MAX_UPLOAD_MB:
        raise ValueError(f'File too large ({size_mb:.1f} MB). Max: {MAX_UPLOAD_MB} MB.')


def load_image_safely(file_path: str) -> PILImage.Image:
    '''Mirrors load_image() — opens, validates format, converts to RGB.'''
    try:
        with PILImage.open(file_path) as img:
            fmt   = img.format
            image = img.convert('RGB')
        if fmt and fmt.upper() not in ALLOWED_FORMATS:
            raise ValueError(f'Unsupported image format: {fmt}')
        return image
    except Exception as exc:
        raise ValueError(f'Cannot open image: {exc}') from exc


def image_to_base64(file_path: str) -> str:
    '''Mirrors image_to_base64() — used by OllamaAnalyzer.'''
    with open(file_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')


def now_iso() -> str:
    '''Mirrors now_iso() — UTC ISO timestamp for metadata.'''
    return datetime.utcnow().isoformat() + 'Z'


def save_image_to_uploads(pil_img: PILImage.Image, original_name: str = '') -> tuple:
    '''Mirrors save_upload() — saves PIL image with UUID filename.'''
    ext      = os.path.splitext(original_name)[1].lower() if original_name else '.jpg'
    ext      = ext if ext in ALLOWED_EXTENSIONS else '.jpg'
    fmt_map  = {'.jpg': 'JPEG', '.jpeg': 'JPEG', '.png': 'PNG', '.webp': 'WEBP'}
    filename = f'{uuid.uuid4().hex}{ext}'
    dest     = os.path.join(UPLOAD_DIR, filename)
    pil_img.save(dest, fmt_map.get(ext, 'JPEG'))
    return dest, filename


print('Image utilities ready.')

In [ ]:
# Cell 4 — CLIP Embedding Service
# Mirrors: backend/services/clip_service.py  (CLIPService)
#        + backend/embeddings/clip_encoder.py (encode_image, encode_pil_image,
#                                              encode_text, _normalize_embedding)

import hashlib
from functools import lru_cache
from typing import List, Tuple

import numpy as np
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor


class CLIPService:
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        # CPU thread optimisation from clip_encoder.py
        if self.device == 'cpu':
            torch.set_num_threads(max(torch.get_num_threads() // 2, 1))
        log.info('Loading CLIP (%s) on %s', CLIP_MODEL_NAME, self.device)
        self.processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
        self.model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
        self.model.to(self.device).eval()
        log.info('CLIP ready')

    def _normalize(self, tensor: torch.Tensor) -> List[float]:
        '''Mirrors _normalize_embedding() from clip_encoder.py.'''
        tensor = tensor / tensor.norm(p=2, dim=-1, keepdim=True)
        return tensor.squeeze(0).cpu().numpy().astype(np.float32).tolist()

    # ── image encoding ────────────────────────────────────────────────────────
    def encode_image(self, image: Image.Image) -> List[float]:
        '''Mirrors encode_pil_image() from clip_encoder.py.'''
        inputs = self.processor(images=image, return_tensors='pt')
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            feat = self.model.get_image_features(**inputs)
        return self._normalize(feat)

    def encode_from_path(self, image_path: str) -> List[float]:
        '''Mirrors encode_image(image_path) from clip_encoder.py.'''
        return self.encode_image(load_image_safely(image_path))

    # ── text encoding ─────────────────────────────────────────────────────────
    def encode_text(self, text: str) -> List[float]:
        if not text or not text.strip():
            raise ValueError('Text must be non-empty')
        text_hash = hashlib.md5(text.strip().lower().encode()).hexdigest()
        return list(self._cached_encode(text_hash, text.strip()))

    @lru_cache(maxsize=256)
    def _cached_encode(self, text_hash: str, text: str) -> Tuple[float, ...]:
        '''LRU-cached text encoding keyed by MD5 hash — from clip_service.py.'''
        inputs = self.processor(text=[text], return_tensors='pt', padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            feat = self.model.get_text_features(**inputs)
        return tuple(self._normalize(feat))

In [ ]:
# Cell 5 — ChromaDB Vector Store
# Full mirror of: backend/services/chroma_service.py
# All methods covered: add_embedding, add_image, search_similar_images,
#   text_search, query, delete_embedding, get_image, count, list_all

import hashlib
from typing import Any, Dict, List, Optional

import chromadb


class ChromaService:
    def __init__(self):
        self.client = chromadb.PersistentClient(path=CHROMA_PATH)
        self.collection = self.client.get_or_create_collection(
            CHROMA_COLLECTION,
            metadata={
                'hnsw:space':           'cosine',
                'hnsw:construction_ef': 200,
                'hnsw:search_ef':       100,
                'hnsw:M':               16,
            },
        )
        log.info('ChromaDB ready — %d images indexed', self.count())

    # ── internal helpers ──────────────────────────────────────────────────────
    def _build_id(self, image_path: str) -> str:
        return hashlib.sha256(image_path.encode('utf-8')).hexdigest()

    def _find_duplicate(self, image_path: str) -> Optional[str]:
        try:
            result = self.collection.get(where={'image_path': image_path})
            ids = result.get('ids', [])
            return ids[0] if ids else None
        except Exception:
            return None

    # ── write operations ──────────────────────────────────────────────────────
    def add_embedding(
        self,
        embedding: List[float],
        image_path: str,
        metadata: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        '''Low-level add — mirrors add_embedding() from chroma_service.py.'''
        existing_id = self._find_duplicate(image_path)
        if existing_id:
            log.info('Duplicate image skipped: %s', image_path)
            existing = self.collection.get(ids=[existing_id], include=['metadatas'])
            return {
                'id':        existing_id,
                'metadata':  (existing.get('metadatas') or [{}])[0],
                'duplicate': True,
            }
        image_id    = self._build_id(image_path)
        record_meta = {'image_path': image_path, 'upload_time': now_iso()}
        if metadata:
            record_meta.update(metadata)
        self.collection.add(
            ids=[image_id],
            embeddings=[embedding],
            metadatas=[record_meta],
            documents=[image_path],
        )
        return {'id': image_id, 'metadata': record_meta, 'duplicate': False}

    def add_image(
        self, embedding: List[float], file_path: str, filename: str
    ) -> Dict[str, Any]:
        '''High-level add — mirrors add_image() from chroma_service.py.'''
        return self.add_embedding(
            embedding, file_path,
            metadata={'filename': filename, 'timestamp': now_iso()},
        )

    def delete_embedding(self, image_id: str) -> bool:
        '''Mirrors delete_embedding() from chroma_service.py.'''
        try:
            self.collection.delete(ids=[image_id])
            log.info('Deleted embedding: %s', image_id[:16])
            return True
        except Exception as exc:
            raise RuntimeError(f'Failed to delete: {exc}') from exc

    # ── read / search operations ──────────────────────────────────────────────
    def search_similar_images(
        self, embedding: List[float], top_k: int
    ) -> List[Dict[str, Any]]:
        '''Mirrors search_similar_images() from chroma_service.py.'''
        total = self.count()
        if total == 0:
            return []
        results = self.collection.query(
            query_embeddings=[embedding],
            n_results=min(top_k, total),
            include=['metadatas', 'distances', 'documents'],
        )
        ids       = results.get('ids',       [[]])[0]
        distances = results.get('distances', [[]])[0]
        metadatas = results.get('metadatas', [[]])[0]
        return [
            {
                'id':         img_id,
                'image_path': meta.get('image_path'),
                'metadata':   meta,
                'distance':   float(dist),
                'similarity': max(0.0, min(1.0, 1.0 - float(dist))),
            }
            for img_id, dist, meta in zip(ids, distances, metadatas)
        ]

    def text_search(
        self, text_embedding: List[float], top_k: int
    ) -> List[Dict[str, Any]]:
        '''Mirrors text_search() alias from chroma_service.py.'''
        return self.search_similar_images(text_embedding, top_k)

    def query(self, embedding: List[float], top_k: int) -> Dict[str, Any]:
        '''Raw ChromaDB query — mirrors query() from chroma_service.py.'''
        n = min(top_k, max(self.count(), 1))
        return self.collection.query(
            query_embeddings=[embedding],
            n_results=n,
            include=['metadatas', 'distances', 'documents'],
        )

    def get_image(self, image_id: str) -> Dict[str, Any]:
        '''Mirrors get_image() from chroma_service.py.'''
        result   = self.collection.get(ids=[image_id], include=['metadatas', 'documents'])
        metadata = (result.get('metadatas') or [{}])[0]
        document = (result.get('documents') or [None])[0]
        return {'id': image_id, 'image_path': document, 'metadata': metadata}

    def count(self) -> int:
        return int(self.collection.count())

    def list_all(self) -> List[Dict[str, Any]]:
        results   = self.collection.get(include=['metadatas'])
        ids       = results.get('ids',       [])
        metadatas = results.get('metadatas', [])
        return [{'id': i, 'metadata': m} for i, m in zip(ids, metadatas)]

In [ ]:
# Cell 6 — Image Analyzers
# Mirrors: backend/services/ollama_service.py  (OllamaService — complete)
#   - analyze_image()     → OllamaAnalyzer.analyze()
#   - healthcheck()       → OllamaAnalyzer.healthcheck()
#   - _infer_confidence() → OllamaAnalyzer._infer_confidence()
#   - retry logic         → OLLAMA_RETRIES attempts with backoff
# BLIP added as a local alternative (no Ollama server required)

import time
from typing import Any, Dict

import requests as _req
import torch
from PIL import Image as PILImage
from transformers import BlipForConditionalGeneration, BlipProcessor

# ── Toggle ────────────────────────────────────────────────────────────────────
# Set USE_OLLAMA = True after running Cell 7 (Ollama install), then re-run
# this cell and Cell 8 so the new analyser is picked up.
USE_OLLAMA = False
# ─────────────────────────────────────────────────────────────────────────────


class BLIPAnalyzer:
    '''Local Salesforce BLIP captioning model — no server required.'''

    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        log.info('Loading BLIP on %s', self.device)
        self.processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
        self.model     = BlipForConditionalGeneration.from_pretrained(
            'Salesforce/blip-image-captioning-base'
        )
        self.model.to(self.device).eval()
        log.info('BLIP ready')

    def analyze(self, image_path: str, prompt: str = '') -> Dict[str, str]:
        image = PILImage.open(image_path).convert('RGB')
        if prompt.strip():
            inputs = self.processor(image, prompt, return_tensors='pt').to(self.device)
        else:
            inputs = self.processor(image, return_tensors='pt').to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=200, num_beams=4)
        analysis = self.processor.decode(out[0], skip_special_tokens=True)
        return {'analysis': analysis, 'confidence': 'medium', 'model': 'BLIP'}


class OllamaAnalyzer:
    '''
    Full mirror of OllamaService from ollama_service.py:
      analyze_image  → analyze()
      healthcheck    → healthcheck()
      _infer_confidence (exact copy)
      retry logic: OLLAMA_RETRIES attempts, exponential backoff (min(attempt, 3)s)
    '''

    def __init__(self):
        self.base_url    = OLLAMA_URL.rstrip('/')
        self.model       = OLLAMA_MODEL
        self.timeout     = OLLAMA_TIMEOUT
        self.max_retries = OLLAMA_RETRIES

    def analyze(self, image_path: str, prompt: str = '') -> Dict[str, str]:
        '''Mirrors analyze_image() from ollama_service.py.'''
        img_b64 = image_to_base64(image_path)
        prompt  = prompt.strip() or 'Describe this image and mention any notable details.'
        payload: Dict[str, Any] = {
            'model':  self.model,
            'prompt': prompt,
            'images': [img_b64],
            'stream': False,
        }
        last_err = None
        for attempt in range(1, self.max_retries + 1):
            try:
                log.info('Ollama model=%s attempt=%d path=%s',
                         self.model, attempt, image_path)
                resp = _req.post(
                    f'{self.base_url}/api/generate',
                    json=payload,
                    timeout=self.timeout,
                )
                resp.raise_for_status()
                data     = resp.json()
                analysis = str(data.get('response', '')).strip()
                if not analysis:
                    raise RuntimeError('Ollama returned an empty analysis')
                return {
                    'analysis':   analysis,
                    'confidence': self._infer_confidence(data, analysis),
                    'model':      'LLaVA',
                }
            except (_req.Timeout, _req.RequestException, ValueError, RuntimeError) as exc:
                last_err = exc
                log.warning('Ollama attempt %d/%d failed: %s', attempt, self.max_retries, exc)
            except Exception as exc:
                last_err = exc
                log.exception('Unexpected Ollama error attempt %d/%d', attempt, self.max_retries)
            if attempt < self.max_retries:
                time.sleep(min(attempt, 3))
        raise RuntimeError(
            f'Ollama analysis failed after {self.max_retries} attempts'
        ) from last_err

    def healthcheck(self) -> None:
        '''Mirrors healthcheck() from ollama_service.py.'''
        try:
            resp = _req.get(f'{self.base_url}/api/tags', timeout=5)
            resp.raise_for_status()
        except _req.RequestException as exc:
            raise RuntimeError('Ollama healthcheck failed') from exc

    def _infer_confidence(self, data: Dict[str, Any], analysis: str) -> str:
        '''Exact mirror of _infer_confidence() from ollama_service.py.'''
        eval_count  = data.get('eval_count')
        done_reason = str(data.get('done_reason', '')).lower()
        if isinstance(eval_count, int) and eval_count >= 150 and done_reason == 'stop':
            return 'high'
        if isinstance(eval_count, int) and eval_count >= 50:
            return 'medium'
        if len(analysis.split()) >= 8:
            return 'medium'
        return 'low'

In [ ]:
# Cell 7 (OPTIONAL) — Install Ollama + pull LLaVA
# ─────────────────────────────────────────────────────────────────────────────
# Skip this cell if BLIP analysis is sufficient.
# Downloads ~4 GB and takes 10-20 minutes.
#
# After completion:
#   1. Go to Cell 6, set USE_OLLAMA = True, re-run Cell 6
#   2. Re-run Cell 8 (Initialize) so the new analyser is picked up
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, time

print('Installing Ollama ...')
subprocess.run(['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)

print('Starting Ollama server ...')
_ollama_proc = subprocess.Popen(
    ['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(5)

print('Pulling LLaVA model (~4 GB, may take 10-20 min) ...')
subprocess.run(['ollama', 'pull', 'llava'], check=True, timeout=1800)
print('Done!  Set USE_OLLAMA = True in Cell 6 and re-run cells 6 & 8.')

In [ ]:
# Cell 8 — Initialize all services
# CLIP download : ~340 MB  (HuggingFace cache — skipped on re-runs)
# BLIP download : ~945 MB  (HuggingFace cache — skipped on re-runs)

print('=' * 55)

clip_service   = CLIPService()
print()
chroma_service = ChromaService()
print()
image_analyzer = OllamaAnalyzer() if USE_OLLAMA else BLIPAnalyzer()

print()
print('=' * 55)
print(f'Analyser  : {"Ollama + LLaVA" if USE_OLLAMA else "BLIP"}')
print(f'DB count  : {chroma_service.count()} images indexed')
print('All services ready!')


# ── Health check  (mirrors GET /health from routes.py) ───────────────────────
def health_check() -> str:
    lines = ['=== Service Health ===', '']
    try:
        clip_service.encode_text('test')
        lines.append('CLIP     : OK')
    except Exception as e:
        lines.append(f'CLIP     : ERROR — {e}')
    try:
        n = chroma_service.count()
        lines.append(f'ChromaDB : OK  ({n} images indexed)')
    except Exception as e:
        lines.append(f'ChromaDB : ERROR — {e}')
    if USE_OLLAMA:
        try:
            image_analyzer.healthcheck()
            lines.append('Ollama   : OK')
        except Exception as e:
            lines.append(f'Ollama   : ERROR — {e}')
    else:
        lines.append('Ollama   : not enabled  (using BLIP)')
    return '\n'.join(lines)


print()
print(health_check())

In [ ]:
# Cell 9 — Core functions
# Mirrors every endpoint in backend/api/routes.py:
#
#   POST /upload-image            → upload_and_index()
#   POST /search-by-text          → text_search()
#   POST /search-text-to-image    → text_search()       (same CLIP+ChromaDB logic)
#   POST /search-by-image         → image_search()
#   POST /search-similar-images   → image_search()      (same CLIP+ChromaDB logic)
#   POST /analyze-image           → analyze_image_fn()
#   GET  /health                  → health_check()      (defined in Cell 8)
#
# Extras (not in REST API):
#   batch_upload()        — index multiple files at once
#   delete_from_index()   — remove one image from DB + disk
#   clear_entire_index()  — wipe all images
#   get_stats()           — summary of the index

import os
import uuid
from typing import Any, Dict, List, Tuple

from PIL import Image as PILImage


# ── helpers ───────────────────────────────────────────────────────────────────
def _pil_from_input(image) -> PILImage.Image:
    if isinstance(image, PILImage.Image):
        return image.convert('RGB')
    import numpy as np
    return PILImage.fromarray(image).convert('RGB')


def normalize_top_k(top_k: int, min_val: int = 1, max_val: int = 50) -> int:
    '''Mirrors normalize_top_k() from routes.py — enforces 1 <= top_k <= 50.'''
    value = int(top_k)
    if value < min_val or value > max_val:
        raise ValueError(f'top_k must be between {min_val} and {max_val}')
    return value


# ── POST /upload-image ────────────────────────────────────────────────────────
def upload_and_index(image) -> Tuple[PILImage.Image, str]:
    if image is None:
        return None, 'No image provided.'
    pil_img        = _pil_from_input(image)
    dest, filename = save_image_to_uploads(pil_img)
    try:
        validate_image_file(dest)
    except ValueError as e:
        os.remove(dest)
        return None, f'Validation error: {e}'
    embedding = clip_service.encode_image(pil_img)
    record    = chroma_service.add_image(embedding, dest, filename)
    status    = 'duplicate (already indexed)' if record.get('duplicate') else 'indexed'
    msg = (
        f'Status : {status}\n'
        f'ID     : {record["id"][:16]}...\n'
        f'Total  : {chroma_service.count()} images indexed'
    )
    return pil_img, msg


# ── Batch upload (extra) ──────────────────────────────────────────────────────
def batch_upload(files) -> str:
    if not files:
        return 'No files selected.'
    file_list = files if isinstance(files, list) else [files]
    results   = []
    for fp in file_list:
        file_path = fp if isinstance(fp, str) else fp.name
        fname = os.path.basename(file_path)
        try:
            validate_image_file(file_path)
            pil_img        = load_image_safely(file_path)
            dest, uname    = save_image_to_uploads(pil_img, fname)
            embedding      = clip_service.encode_image(pil_img)
            record         = chroma_service.add_image(embedding, dest, uname)
            status         = 'duplicate' if record.get('duplicate') else 'indexed'
            results.append(f'OK  {fname}  [{status}]')
        except Exception as exc:
            results.append(f'ERR {fname}  [{exc}]')
    total = chroma_service.count()
    return '\n'.join(results) + f'\n\nTotal indexed: {total}'


# ── POST /search-by-text  &  /search-text-to-image ───────────────────────────
def text_search(query: str, top_k: int = TOP_K_DEFAULT) -> Tuple[List, str]:
    if not query.strip():
        return [], 'Enter a search query.'
    if chroma_service.count() == 0:
        return [], 'Index is empty — upload images first.'
    embedding = clip_service.encode_text(query)
    results   = chroma_service.text_search(embedding, normalize_top_k(int(top_k)))
    items = [
        (r['image_path'], f'Score: {r["similarity"]:.3f}  |  {r["metadata"].get("filename", "")}')
        for r in results
        if r.get('image_path') and os.path.exists(r['image_path'])
    ]
    return items, f'Found {len(items)} result(s) for: "{query}"'


# ── POST /search-by-image  &  /search-similar-images ─────────────────────────
def image_search(query_image, top_k: int = TOP_K_DEFAULT) -> Tuple[List, str]:
    if query_image is None:
        return [], 'Upload a query image.'
    if chroma_service.count() == 0:
        return [], 'Index is empty — upload images first.'
    pil_img   = _pil_from_input(query_image)
    embedding = clip_service.encode_image(pil_img)
    results   = chroma_service.search_similar_images(embedding, normalize_top_k(int(top_k)))
    items = [
        (r['image_path'], f'Score: {r["similarity"]:.3f}  |  {r["metadata"].get("filename", "")}')
        for r in results
        if r.get('image_path') and os.path.exists(r['image_path'])
    ]
    return items, f'Found {len(items)} similar image(s).'


# ── POST /analyze-image ───────────────────────────────────────────────────────
def analyze_image_fn(image, prompt: str = '') -> str:
    if image is None:
        return 'Upload an image to analyze.'
    pil_img  = _pil_from_input(image)
    tmp_path = os.path.join(UPLOAD_DIR, f'_tmp_{uuid.uuid4().hex}.jpg')
    pil_img.save(tmp_path, 'JPEG')
    try:
        result = image_analyzer.analyze(tmp_path, prompt)
        model  = result.get('model', 'unknown')
        conf   = result.get('confidence', '-')
        return f'[{model} | confidence: {conf}]\n\n{result["analysis"]}'
    except Exception as exc:
        return f'Analysis failed: {exc}'
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


# ── Index management ──────────────────────────────────────────────────────────
def get_indexed_filenames() -> List[str]:
    return [e['metadata'].get('filename', 'unknown') for e in chroma_service.list_all()]


def delete_from_index(filename: str):
    import gradio as gr
    if not filename:
        return 'No image selected.', gr.Dropdown(choices=get_indexed_filenames())
    entries = chroma_service.list_all()
    target  = next((e for e in entries if e['metadata'].get('filename') == filename), None)
    if not target:
        return f'Not found: {filename}', gr.Dropdown(choices=get_indexed_filenames())
    chroma_service.delete_embedding(target['id'])
    img_path = target['metadata'].get('image_path', '')
    if img_path and os.path.exists(img_path):
        os.remove(img_path)
    remaining = get_indexed_filenames()
    return (
        f'Deleted: {filename}\nTotal remaining: {chroma_service.count()}',
        gr.Dropdown(choices=remaining, value=None),
    )


def clear_entire_index():
    import gradio as gr
    entries = chroma_service.list_all()
    count   = 0
    for e in entries:
        try:
            chroma_service.delete_embedding(e['id'])
            ip = e['metadata'].get('image_path', '')
            if ip and os.path.exists(ip):
                os.remove(ip)
            count += 1
        except Exception:
            pass
    return f'Cleared {count} images.', gr.Dropdown(choices=[])


def get_stats() -> str:
    total   = chroma_service.count()
    entries = chroma_service.list_all()
    recent  = [e['metadata'].get('filename', '?') for e in entries[-10:]]
    lines   = [f'Total indexed : {total}', '', 'Recent uploads (last 10):']
    lines  += ([f'  {f}' for f in reversed(recent)] if recent else ['  (none)'])
    return '\n'.join(lines)


print('All core functions ready.')
print(f'DB size: {chroma_service.count()} images')

In [ ]:
# Cell 10 — Gradio UI
# Tabs mirror every route in api/routes.py + extras:
#   Tab 1  Upload           POST /upload-image
#   Tab 2  Batch Upload     (multiple files at once)
#   Tab 3  Text Search      POST /search-by-text + /search-text-to-image
#   Tab 4  Image Search     POST /search-by-image + /search-similar-images
#   Tab 5  Analyse Image    POST /analyze-image
#   Tab 6  Manage Index     delete, clear, health check, stats

import gradio as gr

_analyser_label = 'LLaVA (Ollama)' if USE_OLLAMA else 'BLIP'

with gr.Blocks(
    title='Multi-Image RAG',
    theme=gr.themes.Soft(),
    css='.gradio-container { max-width: 1150px !important; }',
) as demo:

    gr.Markdown(
        '# Multi-Image RAG\n'
        f'**CLIP + ChromaDB + {_analyser_label}** — semantic image search & AI analysis'
    )

    with gr.Tabs():

        # ── Tab 1: Upload ──────────────────────────────────────────────────────
        with gr.Tab('Upload & Index'):
            gr.Markdown('Upload one image at a time — validates format & size (max 10 MB).')
            with gr.Row():
                with gr.Column(scale=1):
                    up_input  = gr.Image(label='Upload Image', type='pil')
                    up_btn    = gr.Button('Upload & Index', variant='primary')
                    stats_btn = gr.Button('Show Index Stats')
                with gr.Column(scale=1):
                    up_preview = gr.Image(label='Indexed Image')
                    up_status  = gr.Textbox(label='Status', lines=3)
                    stats_out  = gr.Textbox(label='Index Stats', lines=10)
            up_btn.click(upload_and_index, inputs=up_input, outputs=[up_preview, up_status])
            stats_btn.click(get_stats, outputs=stats_out)

        # ── Tab 2: Batch Upload ────────────────────────────────────────────────
        with gr.Tab('Batch Upload'):
            gr.Markdown('Select multiple images at once — all are validated and indexed.')
            batch_input  = gr.File(
                label='Select images (PNG / JPEG / WEBP)',
                file_count='multiple',
                file_types=['.png', '.jpg', '.jpeg', '.webp'],
            )
            batch_btn    = gr.Button('Upload All & Index', variant='primary')
            batch_status = gr.Textbox(label='Results', lines=15)
            batch_btn.click(batch_upload, inputs=batch_input, outputs=batch_status)

        # ── Tab 3: Text Search ─────────────────────────────────────────────────
        with gr.Tab('Text Search'):
            gr.Markdown(
                'Natural-language query → CLIP embedding → ChromaDB cosine search.\n'
                '*Mirrors* `POST /search-by-text` and `POST /search-text-to-image`.'
            )
            with gr.Row():
                with gr.Column(scale=3):
                    txt_query = gr.Textbox(
                        label='Search query',
                        placeholder='e.g.  a golden retriever running on a beach',
                    )
                with gr.Column(scale=1):
                    txt_topk = gr.Slider(1, 50, value=5, step=1, label='top_k (1–50)')
            txt_btn     = gr.Button('Search', variant='primary')
            txt_status  = gr.Textbox(show_label=False, lines=1)
            txt_gallery = gr.Gallery(label='Results', columns=3, height=480, object_fit='contain')
            txt_btn.click(
                text_search, inputs=[txt_query, txt_topk], outputs=[txt_gallery, txt_status]
            )
            txt_query.submit(
                text_search, inputs=[txt_query, txt_topk], outputs=[txt_gallery, txt_status]
            )

        # ── Tab 4: Image Search ────────────────────────────────────────────────
        with gr.Tab('Image Search'):
            gr.Markdown(
                'Upload a query image → CLIP embedding → ChromaDB cosine search.\n'
                '*Mirrors* `POST /search-by-image` and `POST /search-similar-images`.'
            )
            with gr.Row():
                with gr.Column(scale=1):
                    img_input  = gr.Image(label='Query Image', type='pil')
                    img_topk   = gr.Slider(1, 50, value=5, step=1, label='top_k (1–50)')
                    img_btn    = gr.Button('Find Similar', variant='primary')
                    img_status = gr.Textbox(show_label=False, lines=1)
                with gr.Column(scale=2):
                    img_gallery = gr.Gallery(
                        label='Similar Images', columns=3, height=480, object_fit='contain'
                    )
            img_btn.click(
                image_search, inputs=[img_input, img_topk], outputs=[img_gallery, img_status]
            )

        # ── Tab 5: Analyse Image ───────────────────────────────────────────────
        with gr.Tab('Analyse Image'):
            gr.Markdown(
                f'Analyse with **{_analyser_label}**.  '
                '*Mirrors* `POST /analyze-image`.  '
                'Leave prompt blank for auto-description.'
            )
            with gr.Row():
                with gr.Column(scale=1):
                    ana_input  = gr.Image(label='Image', type='pil')
                    ana_prompt = gr.Textbox(
                        label='Custom prompt (optional)',
                        placeholder='e.g.  What objects are visible? Describe any defects.',
                        lines=2,
                    )
                    ana_btn    = gr.Button('Analyse', variant='primary')
                with gr.Column(scale=1):
                    ana_output = gr.Textbox(label='Analysis', lines=18)
            ana_btn.click(
                analyze_image_fn, inputs=[ana_input, ana_prompt], outputs=ana_output
            )

        # ── Tab 6: Manage Index ────────────────────────────────────────────────
        with gr.Tab('Manage Index'):
            gr.Markdown(
                'Delete images from the index, check service health, and view stats.'
            )
            with gr.Row():
                with gr.Column():
                    refresh_btn  = gr.Button('Refresh Image List')
                    del_dropdown = gr.Dropdown(
                        choices=get_indexed_filenames(),
                        label='Select image to delete',
                        interactive=True,
                    )
                    with gr.Row():
                        del_btn   = gr.Button('Delete Selected', variant='stop')
                        clear_btn = gr.Button('Clear Entire Index', variant='stop')
                    del_status = gr.Textbox(label='Status', lines=4)
                with gr.Column():
                    health_btn = gr.Button('Run Health Check')
                    health_out = gr.Textbox(label='Health', lines=6)
                    stats_btn2 = gr.Button('Show Stats')
                    stats_out2 = gr.Textbox(label='Stats', lines=10)

            refresh_btn.click(
                lambda: gr.Dropdown(choices=get_indexed_filenames()), outputs=del_dropdown
            )
            del_btn.click(
                delete_from_index, inputs=del_dropdown, outputs=[del_status, del_dropdown]
            )
            clear_btn.click(clear_entire_index, outputs=[del_status, del_dropdown])
            health_btn.click(health_check, outputs=health_out)
            stats_btn2.click(get_stats, outputs=stats_out2)

print('Gradio UI defined. Run Cell 11 to launch.')

In [ ]:
# Cell 11 — Launch Gradio
# share=True generates a public tunnel URL — open it in any browser.
# The link expires after 72 hours; re-run this cell to get a fresh one.

demo.launch(share=True)